# Hoboken Citi Bike shortage prediction baseline

This notebook uses collected history only. If there is not enough history yet, it shows the preparation pipeline and stops before reporting any model result.

In [ ]:
from pathlib import Path
import pandas as pd

HISTORY_DIR = Path('data/history')  # In Colab, upload or mount this folder first.
history_files = sorted(HISTORY_DIR.glob('*.csv'))
if not history_files:
    print('No history CSV files found. Run scripts/collect_snapshot.py before training a model.')
    history = pd.DataFrame()
else:
    history = pd.concat([pd.read_csv(path) for path in history_files], ignore_index=True)
    print(f'Loaded {len(history):,} rows from {len(history_files)} daily files.')

In [ ]:
def prepare_prediction_frame(history):
    frame = history.copy()
    frame['timestamp_utc'] = pd.to_datetime(frame['timestamp_utc'], utc=True, errors='coerce')
    frame = frame.dropna(subset=['timestamp_utc', 'station_id']).sort_values(['station_id', 'timestamp_utc'])
    frame['hour_of_day'] = frame['timestamp_utc'].dt.hour
    frame['day_of_week'] = frame['timestamp_utc'].dt.dayofweek
    results = []
    for _, station in frame.groupby('station_id', group_keys=False):
        station = station.sort_values('timestamp_utc').copy()
        station['bike_change_15m'] = station['bikes_available'].diff()
        station['dock_change_15m'] = station['docks_available'].diff()
        station['bike_ratio_rolling_60m'] = station['bike_availability_ratio'].rolling(4, min_periods=1).mean()
        station['dock_ratio_rolling_60m'] = station['dock_availability_ratio'].rolling(4, min_periods=1).mean()
        for minutes, steps in [(15, 1), (30, 2), (60, 4)]:
            future_time = station['timestamp_utc'].shift(-steps)
            interval_ok = (future_time - station['timestamp_utc']).dt.total_seconds().between(minutes * 60 - 480, minutes * 60 + 480)
            station[f'bikes_plus_{minutes}m'] = station['bikes_available'].shift(-steps).where(interval_ok)
            station[f'docks_plus_{minutes}m'] = station['docks_available'].shift(-steps).where(interval_ok)
        station['bike_shortage_60m'] = (station['bikes_plus_60m'] <= 2).where(station['bikes_plus_60m'].notna())
        station['dock_shortage_60m'] = (station['docks_plus_60m'] <= 2).where(station['docks_plus_60m'].notna())
        results.append(station)
    return pd.concat(results, ignore_index=True) if results else frame

features = prepare_prediction_frame(history) if not history.empty else pd.DataFrame()

In [ ]:
if not features.empty:
    print('Rows:', len(features))
    print('Stations:', features['station_id'].nunique())
    print('Date range:', features['timestamp_utc'].min(), 'to', features['timestamp_utc'].max())
    print('Bike shortage class balance:', features['bike_shortage_60m'].value_counts(dropna=False).to_dict())
    print('Dock shortage class balance:', features['dock_shortage_60m'].value_counts(dropna=False).to_dict())

In [ ]:
feature_columns = ['bike_availability_ratio', 'dock_availability_ratio', 'bike_change_15m', 'dock_change_15m', 'bike_ratio_rolling_60m', 'dock_ratio_rolling_60m', 'hour_of_day', 'day_of_week', 'weather_friction', 'temperature', 'nearest_relevant_construction_distance_m']
training = features.dropna(subset=['bike_shortage_60m']).copy() if not features.empty else pd.DataFrame()
if len(training) < 100 or training['bike_shortage_60m'].nunique() < 2:
    print('Not enough valid history for a trustworthy model yet. Keep collecting 15-minute snapshots across more days and different conditions.')
else:
    split_index = int(len(training) * 0.8)
    train = training.iloc[:split_index]
    test = training.iloc[split_index:]
    print(f'Time-based split: {len(train)} train rows, {len(test)} future test rows.')

In [ ]:
if 'train' in globals():
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score
    from sklearn.pipeline import Pipeline

    X_train, X_test = train[feature_columns], test[feature_columns]
    y_train, y_test = train['bike_shortage_60m'].astype(int), test['bike_shortage_60m'].astype(int)
    persistence = (test['bikes_available'] <= 2).astype(int)
    models = {
        'persistence': persistence,
        'logistic regression': Pipeline([('impute', SimpleImputer(strategy='median')), ('model', LogisticRegression(max_iter=500, class_weight='balanced'))]),
        'random forest': Pipeline([('impute', SimpleImputer(strategy='median')), ('model', RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced', n_jobs=-1))]),
    }
    for name, model in models.items():
        prediction = model if name == 'persistence' else model.fit(X_train, y_train).predict(X_test)
        print(name, {'precision': round(precision_score(y_test, prediction, zero_division=0), 3), 'recall': round(recall_score(y_test, prediction, zero_division=0), 3), 'f1': round(f1_score(y_test, prediction, zero_division=0), 3)})
        ConfusionMatrixDisplay.from_predictions(y_test, prediction)
        if name != 'persistence' and y_test.nunique() == 2:
            probabilities = model.predict_proba(X_test)[:, 1]
            print('ROC-AUC:', round(roc_auc_score(y_test, probabilities), 3))
    # Save a model only after reviewing valid results, for example with joblib.dump(...).

Repeat the same time-based process for `dock_shortage_60m`. Do not use a random split because it lets future time patterns leak into training.